In [ ]:
'''data extraction'''

import pandas as pd
import numpy as np

f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

training = pd.concat([kazuki_data, yokoi_data.loc[0,:].iloc[:2000], yokoi_data.loc[1,:].iloc[:200]])
validation = pd.concat([watara_data, yokoi_data.loc[0,:].iloc[2000:], yokoi_data.loc[1,:].iloc[200:]])

t_healthy = training.loc[0]
t_cancer = training.loc[1]
v_healthy = validation.loc[0]
v_cancer = validation.loc[1]

m_t_healthy = t_healthy.shape[0]
m_t_cancer = t_cancer.shape[0]
m_v_healthy = v_healthy.shape[0]
m_v_cancer = v_cancer.shape[0]

n_mirnas = t_healthy.shape[1] #gets number of columns
miRNA_names_list = list(t_healthy.columns)

print("healthy training data")
print(t_healthy.shape)
print(t_healthy.head)
print("cancer training data")
print(t_cancer.shape)
print(t_cancer.head)
print("healthy validation data")
print(v_healthy.shape)
print(v_healthy.head)
print("cancer validation data")
print(v_cancer.shape)
print(v_cancer.head)

healthy training data
(6965, 2565)
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  ...  MIMAT0035703  MIMAT0035704
0          3.700         3.669  ...         2.795        -2.681
0         -4.018        -4.018  ...        -4.018        -4.018
0          1.514         0.869  ...        -4.325        -4.325
0         -4.417        -4.417  ...        -1.197        -4.417
0         -5.541         0.598  ...        -4.377        -3.371
..           ...           ...  ...           ...           ...
0         -3.515         1.626  ...        -3.515        -3.515
0         -3.160        -3.160  ...        -3.160        -3.160
0         -3.717         0.509  ...        -3.717        -3.717
0         -2.374        -2.374  ...         1.840        -2.374
0         -5.741         1.007  ...        -5.741        -5.741

[6965 rows x 2565 columns]>
cancer training data
(200, 2565)
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  ...  MIMAT0035703  MIMAT0035704
1          

In [ ]:
'''Y = training cancer dataset. 
healthy comp is nxn matrix of 
comparison of expression of Ea>Eb'''
Y = t_healthy.values
healthy_comp = np.zeros((n_mirnas, n_mirnas))
for a in range(n_mirnas):
    for b in range(n_mirnas):
        healthy_comp[a,b]=np.sum(Y[:,a] > Y[:,b])
healthy_comp = healthy_comp / m_t_healthy
print(healthy_comp)

NameError: name 't_healthy' is not defined

In [ ]:
'''decide on stable pairs from healthy'''
CONST_k = 0.95
stable_pairs = np.where((healthy_comp >= CONST_k) | (healthy_comp <= 1-CONST_k), 
                        1, 0)

NameError: name 'np' is not defined

In [5]:
print(stable_pairs.nonzero()[0].shape)

(2563428,)


In [ ]:
'''Z = training cancer dataset. 
cancer comp is nxn matrix of 
comparison of expression of Ea>Eb'''
Z = t_cancer.values
cancer_comp = np.zeros((n_mirnas, n_mirnas))
for a in range(n_mirnas):
    for b in range(n_mirnas):
        cancer_comp[a,b]=np.sum(Z[:,a] > Z[:,b])
cancer_comp = cancer_comp / m_t_cancer
print(cancer_comp)

[[0.    0.45  0.525 ... 0.505 0.67  0.635]
 [0.4   0.    0.505 ... 0.475 0.645 0.665]
 [0.29  0.31  0.    ... 0.38  0.52  0.49 ]
 ...
 [0.4   0.43  0.455 ... 0.    0.59  0.56 ]
 [0.15  0.13  0.18  ... 0.145 0.    0.205]
 [0.14  0.125 0.215 ... 0.23  0.385 0.   ]]


In [ ]:
'''A,B are healthy/ovarian training
A,B are healthy/ovarian validation 
bc X is used'''
A = v_healthy.values
B = v_cancer.values

[[0.         0.16666667 0.26801802 ... 0.10810811 0.3795045  0.38063063]
 [0.63626126 0.         0.64752252 ... 0.17567568 0.73198198 0.73873874]
 [0.31869369 0.1768018  0.         ... 0.12274775 0.40202703 0.40653153]
 ...
 [0.85810811 0.79617117 0.84459459 ... 0.         0.92004505 0.91891892]
 [0.10810811 0.05968468 0.11373874 ... 0.01801802 0.         0.13851351]
 [0.08333333 0.03490991 0.07545045 ... 0.02364865 0.12162162 0.        ]]
[[0.      0.33125 0.425   ... 0.5     0.61875 0.6125 ]
 [0.525   0.      0.55    ... 0.55    0.73125 0.725  ]
 [0.4     0.325   0.      ... 0.50625 0.68125 0.6625 ]
 ...
 [0.33125 0.29375 0.35625 ... 0.      0.5875  0.5375 ]
 [0.04375 0.025   0.05    ... 0.0625  0.      0.1125 ]
 [0.1     0.0875  0.08125 ... 0.1625  0.24375 0.     ]]


In [8]:
print(A.shape, B.shape)

(888, 2565) (160, 2565)


In [ ]:
'''you can figure this out. '''
CONST_l = 0.75
stable_cancer_pairs = np.where((cancer_comp >= CONST_l) | (cancer_comp <= 1-CONST_l), 
                        1, 0)

In [10]:
identity_matrix = np.identity(n_mirnas,dtype=bool)
candidate_diagnostic_pairs = (((stable_pairs) & (stable_cancer_pairs)) & 
                            ((np.absolute(healthy_comp - cancer_comp)) > 0.5) &
                            (~identity_matrix))



In [11]:
'''
you might consider getting rid of redundnacy here.
check for correlation of clusters
'''
selected = candidate_diagnostic_pairs.nonzero()
# double check ts actually passes 
# for i in range(2):
#     for j in range(len(selected[i])):
#         print(miRNA_names_list[selected[i][j]] in og_set1 or miRNA_names_list[selected[i][j]] in og_set2)
# for i in range(len(selected[0])):
#     x = (t_healthy.iloc[:, selected[0][i]] > t_healthy.iloc[:, selected[1][i]]).sum()
#     print(f'mirna {selected[0][i]} and {selected[1][i]}')
print(selected[0].shape)
counts = np.bincount(selected[0])
for i, count in enumerate(counts):
    if count > 0:
        print(f"{i} occurs {count} times")


(485,)
9 occurs 2 times
38 occurs 1 times
45 occurs 1 times
71 occurs 1 times
135 occurs 7 times
148 occurs 1 times
328 occurs 1 times
334 occurs 1 times
344 occurs 1 times
345 occurs 4 times
374 occurs 1 times
412 occurs 3 times
461 occurs 1 times
489 occurs 1 times
514 occurs 1 times
535 occurs 1 times
542 occurs 14 times
566 occurs 2 times
589 occurs 1 times
595 occurs 2 times
616 occurs 1 times
626 occurs 1 times
630 occurs 1 times
673 occurs 2 times
688 occurs 2 times
694 occurs 1 times
696 occurs 1 times
704 occurs 1 times
713 occurs 12 times
714 occurs 5 times
731 occurs 7 times
736 occurs 1 times
745 occurs 18 times
762 occurs 4 times
765 occurs 1 times
785 occurs 1 times
791 occurs 3 times
810 occurs 5 times
815 occurs 1 times
829 occurs 1 times
836 occurs 2 times
872 occurs 5 times
879 occurs 2 times
913 occurs 1 times
926 occurs 1 times
931 occurs 1 times
933 occurs 9 times
939 occurs 1 times
944 occurs 1 times
957 occurs 1 times
965 occurs 16 times
966 occurs 1 times
970 oc

In [12]:
'''generate training input'''
pair_dict_training = {}
for i in range(len(selected[0])):
    _key = f"{selected[0][i]},{selected[1][i]}"
    _healthy_sect = Y[:,selected[0][i]]> Y[:,selected[1][i]]
    _cancer_sect = Z[:,selected[0][i]]> Z[:,selected[1][i]]
    pair_dict_training[_key] = np.concat((_healthy_sect, _cancer_sect))
_index = np.concat((np.zeros(m_t_healthy),np.ones(m_t_cancer)))
X_unshuffled = pd.DataFrame(pair_dict_training, index = _index)
X = X_unshuffled.sample(frac=1, random_state=42)
y = np.array(list(X.index))
X = X.reset_index().drop('index', axis=1)

In [13]:
'''generate validation input'''
pair_dict_validation = {}
for i in range(len(selected[0])):
    _key = f"{selected[0][i]},{selected[1][i]}"
    _healthy_sect = A[:,selected[0][i]]> A[:,selected[1][i]]
    _cancer_sect = B[:,selected[0][i]]> B[:,selected[1][i]]
    pair_dict_validation[_key] = np.concat((_healthy_sect, _cancer_sect))
_index = np.concat((np.zeros(m_v_healthy),np.ones(m_v_cancer)))
Xv_unshuffled = pd.DataFrame(pair_dict_validation, index = _index)
Xv = Xv_unshuffled.sample(frac=1, random_state=42)
yv = np.array(list(Xv.index))
Xv = Xv.reset_index().drop('index', axis=1)

In [14]:
X.shape, Xv.shape, y.shape, yv.shape

((7165, 485), (1048, 485), (7165,), (1048,))

In [15]:
Xv

,"9,374","9,731","38,1430","45,973","71,542","135,688","135,1600","135,2189","135,2441","135,2492","135,2523","135,2541","148,2241","328,1748","334,2114","344,731","345,630","345,713","345,1101","345,1909","374,9","412,1123","412,1773","412,1905","461,1748","489,745","514,745","535,713","542,38","542,71","542,554","542,617","542,815","542,926","542,973","542,1181","542,1242","542,1601","542,1926","542,2114",...,"2419,2525","2423,1908","2427,745","2435,872","2440,2114","2441,135","2441,765","2441,1065","2441,1499","2441,1908","2443,1908","2443,2525","2448,713","2449,1908","2449,2525","2480,1347","2492,135","2492,1328","2492,1494","2492,1574","2521,1123","2521,1773","2521,1905","2523,135","2523,1310","2525,673","2525,836","2525,872","2525,1139","2525,1245","2525,1296","2525,1419","2525,1748","2525,2241","2525,2281","2525,2419","2525,2443","2525,2449","2541,135","2560,713"
0,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,True,True,True,True,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,...,True,True,True,False,True,True,True,True,True,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,True,True,True,False,True,True,True,True,True,True,True,True,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,True,True,True,True,True,True,True,True,False,False
2,True,True,True,False,True,False,False,True,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,True,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,False,False
3,False,True,True,False,True,True,False,True,False,True,True,True,True,True,True,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,True,True,True,True,True,True,True,True,False,False
4,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,True,True,True,True,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,...,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1043,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,True,True,True,True,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,...,True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True
1044,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,True,

In [16]:
np.sum(yv)

np.float64(160.0)

In [102]:
#logistic regression with lasso
import matplotlib.pyplot as plt
import numpy as np

from sklearn.svm import SVC
from sklearn.feature_selection import RFE, RFECV
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import BaggingClassifier

import scipy.stats

import matplotlib.pyplot
# not importing scaling bc all of it is binary

In [ ]:
#increase tolerance for short training time ? idk what ts means
C = [1.0,0.1,0.01]
class_weight = ['balanced', None]

OV_PREV = 0.0004

graph_data = {'sensitivity': [], 'specificity': [], 'ppv': [], 'npv': []}
svc_graph_data = {'sensitivity': [], 'specificity': [], 'ppv': [], 'npv': []}

for c in range(len(C)):
    if(c==2):
        pass
    for key in graph_data:
        graph_data[key].append([])
        svc_graph_data[key].append([])
    for cw in range(len(class_weight)):
        svc_loop = SVC(C=C[c], class_weight=class_weight[cw], kernel='linear')
        svc_loop.fit(X,y)

        model_loop = RFE(estimator=svc_loop, n_features_to_select=13,step=1)
        model_loop.fit(X,y)
        yv_pred_loop = model_loop.decision_function(Xv)
        yv_prediction_values_loop = np.array(yv_pred_loop)
        
        cmv_loop = confusion_matrix(yv, yv_prediction_values_loop>0)
        sensitivityv_loop = cmv_loop[1,1]/(cmv_loop[1,1]+cmv_loop[1,0])
        specificityv_loop = (cmv_loop[0,0]/(cmv_loop[0,0]+cmv_loop[0,1]))

        ppv_loop = (sensitivityv_loop*OV_PREV) / ((sensitivityv_loop*OV_PREV) + (1-specificityv_loop) * (1-OV_PREV))
        npv_loop = (specificityv_loop * (1-OV_PREV) / ((specificityv_loop*(1-OV_PREV)) + (1-sensitivityv_loop) * OV_PREV))
        
        graph_data['sensitivity'][c].append(sensitivityv_loop)
        graph_data['specificity'][c].append(specificityv_loop)
        graph_data['ppv'][c].append(ppv_loop)
        graph_data['npv'][c].append(npv_loop)
        
        # ----- cmv for the svc
        # yv_prediction_values_loop = svc_loop.predict(Xv) > 0.5
        # cmv_loop = confusion_matrix(yv, yv_prediction_values_loop>0)
        # sensitivityv_loop = cmv_loop[1,1]/(cmv_loop[1,1]+cmv_loop[1,0])
        # specificityv_loop = (cmv_loop[0,0]/(cmv_loop[0,0]+cmv_loop[0,1]))

        # ppv_loop = (sensitivityv_loop*OV_PREV) / ((sensitivityv_loop*OV_PREV) + (1-specificityv_loop) * (1-OV_PREV))
        # npv_loop = (specificityv_loop * (1-OV_PREV) / ((specificityv_loop*(1-OV_PREV)) + (1-sensitivityv_loop) * OV_PREV))

        # svc_graph_data['sensitivity'][c].append(sensitivityv_loop)
        # svc_graph_data['specificity'][c].append(specificityv_loop)
        # svc_graph_data['ppv'][c].append(ppv_loop)
        # svc_graph_data['npv'][c].append(npv_loop)
graph_data

{'sensitivity': [[np.float64(0.9375), np.float64(0.9375)],
  [np.float64(0.9375), np.float64(0.9375)],
  [np.float64(0.975), np.float64(0.91875)]],
 'specificity': [[np.float64(0.9684684684684685),
   np.float64(0.9684684684684685)],
  [np.float64(0.9684684684684685), np.float64(0.9684684684684685)],
  [np.float64(0.9673423423423423), np.float64(0.9673423423423423)]],
 'ppv': [[np.float64(0.01175772726309768), np.float64(0.01175772726309768)],
  [np.float64(0.01175772726309768), np.float64(0.01175772726309768)],
  [np.float64(0.011805805543737911), np.float64(0.011132283622505247)]],
 'npv': [[np.float64(0.9999741763836747), np.float64(0.9999741763836747)],
  [np.float64(0.9999741763836747), np.float64(0.9999741763836747)],
  [np.float64(0.9999896583683925), np.float64(0.9999663904793242)]]}

In [ ]:
fig,ax = plt.subplots(nrows=2, ncols=len(2), sharey=True)
graph_titles = [['sensitivity', 'specificity'], ['ppv', 'npv']]
for i, row in enumerate(ax):
    for j, col in enumerate(row):
        title = graph_titles[i][j]
        col.bar()

In [86]:
svc = SVC(C=0.1,kernel='linear',class_weight="balanced")
svc.fit(X,y)

,C,0.1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [ ]:
'''sklearn.svm.SVC(kernel = 'poly', degree=3), #default
sklearn.svm.SVC(kernel = 'rbf'),
sklearn.svm.SVC(kernel = 'sigmoid')'''
param_grid = {'C': scipy.stats.expon(scale=100), 
               'class_weight':['balanced',None]}
RandomSearchCV = RandomizedSearchCV(estimator=svc, param_distributions=param_grid, 
                                    n_iter=50, random_state=42)
RandomSearchCV.fit(X,y)
best_params = RandomSearchCV.get_params()
print(best_params) #i just changed the og svc to the params given

In [ ]:
model = RFE(estimator=svc, n_features_to_select=10, step=1)
model = RFECV(estimator=svc, min_features_to_select = 1, step=1)
model.fit(X,y)

y_pred = model.decision_function(X)
y_prediction_values = np.array(y_pred)
cm = confusion_matrix(y, y_prediction_values>0)
sensitivity = cm[1,1]/(cm[1,1]+cm[1,0])
specificity = (cm[0,0]/(cm[0,0]+cm[0,1]))
print("Confusion Matrix: \n", cm)
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")

print(classification_report(y,y_prediction_values>0))


Confusion Matrix: 
 [[6965    0]
 [   0  200]]
Sensitivity (Recall): 1.0000
Specificity: 1.0000
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      6965
         1.0       1.00      1.00      1.00       200

    accuracy                           1.00      7165
   macro avg       1.00      1.00      1.00      7165
weighted avg       1.00      1.00      1.00      7165



In [107]:
yv_pred = model.decision_function(Xv)
yv_prediction_values = np.array(yv_pred)
cmv = confusion_matrix(yv, yv_prediction_values>0)
sensitivityv = cmv[1,1]/(cmv[1,1]+cmv[1,0])
specificityv = (cmv[0,0]/(cmv[0,0]+cmv[0,1]))
print("Confusion Matrix: \n", cmv)
print(f"Sensitivity (Recall): {sensitivityv:.4f}")
print(f"Specificity: {specificityv:.4f}")
print(classification_report(yv,yv_prediction_values>0))

Confusion Matrix: 
 [[860  28]
 [  7 153]]
Sensitivity (Recall): 0.9563
Specificity: 0.9685
              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98       888
         1.0       0.85      0.96      0.90       160

    accuracy                           0.97      1048
   macro avg       0.92      0.96      0.94      1048
weighted avg       0.97      0.97      0.97      1048



In [100]:
bag_model = BaggingClassifier(estimator=model, n_estimators = 10, random_state=42)
bag_model.fit(X,y)

bag_model.estimators_[0].ranking_


array([359,  79, 268, 154, 220, 251, 118, 159, 177, 271, 201, 411,   1,
         3, 469, 343,  25, 282, 281,  26, 184, 351, 459, 420,  36,  97,
       463, 344, 246, 307, 133, 475, 386, 399, 354, 387, 264, 283, 192,
       403, 137, 162, 418, 439,  71, 291,  48, 232,  34,  65, 231,   5,
       286, 301, 252, 378,   6, 297, 323,   9,  63, 276,  47,  84, 350,
       345, 417,  30, 158,  49, 114, 476, 207,  82,  13, 452, 330, 228,
       328, 238,  66, 385, 108, 461, 257, 144, 265, 258, 267, 401,  16,
       260, 167, 136, 451, 186, 163, 214, 255,   1, 369, 432, 470, 465,
       464,  80,   1,  55,  46, 319, 208, 170, 109,  22, 150, 174, 298,
       412,   8, 410, 341, 315, 152, 364, 182, 400, 263, 235,  27, 172,
        53,  10, 185, 122, 324,  21,  89,  96, 219,  31, 142,  94, 128,
       380, 449, 444, 340, 126, 373,  35, 262, 454, 347, 237, 374, 110,
        69,  72, 338, 195, 306, 275, 352, 356, 248, 367, 157, 171,  17,
        77,  12,  85, 138, 280,  29, 389, 365,  58, 366, 203, 34

In [38]:
avg_ranking = np.concatenate(tuple([np.reshape(bag_model.estimators_[i].ranking_, (-1,1)) for i in range(10)]), axis=1)
avg_ranking

array([[359, 219, 368, ..., 227, 169, 369],
       [103, 395, 298, ..., 407, 263, 324],
       [268,  86, 292, ..., 321, 253, 316],
       ...,
       [462, 113,  98, ...,  25, 133, 454],
       [388, 433, 227, ..., 192, 242, 223],
       [421, 131, 153, ..., 149, 252, 465]], shape=(485, 10))

In [ ]:
avg_ranking = np.sum(avg_ranking, axis=1) / 10

array([329.4, 303.1, 230.8, 228.5, 183.3, 345.4, 370.1, 238.9, 302.8,
       152.6, 293.5, 332.2, 143.1, 160. , 312.3, 261.8, 214.8, 333.6,
       347.5, 169.3, 355.3, 271.9, 319. , 258.3,  49.6, 239.7, 372.9,
       258.1, 273.1, 265.8, 143.1, 326.2, 315.8, 306.8, 257.6, 212.4,
       205.4, 265.5, 315.1, 286.8, 318. , 327.5, 290.3, 290.8,  68.3,
       343.3, 139.6, 299.6,  18.2, 208.4, 236.3,  85.6, 366.3, 249.1,
       295.8, 285.8,  73.5, 343.9, 237.6,  76.9, 150.9, 287.2,  39.3,
       339.4, 208.2, 331.7, 290.9, 100.6, 164.6, 173.3, 166.3, 285.7,
       279.8, 204.9, 235.9, 243.4, 272.8, 101.1, 149.2, 265.9,  63.1,
       294.4, 236.9, 363.9, 342.2, 294.2, 278.9, 331.5, 156.5, 358.5,
        29.7, 206. , 335.4, 301.7, 357. , 149.2, 115.4, 203.3, 158.3,
        10.7, 235. , 263.1, 284. , 288.3, 340.9, 297.3, 105.7,  16.2,
        27.3, 262. , 246.6, 335.4, 300.8,  47.5, 193.5, 295.6, 329.3,
       271.5, 149. , 336.6, 300.2, 241.2, 283.9, 342.2, 319. , 195.9,
       317.2, 247.5,

In [43]:
y_pred = bag_model.decision_function(X)
y_prediction_values = np.array(y_pred)
cm = confusion_matrix(y, y_prediction_values>0)
sensitivity = cm[1,1]/(cm[1,1]+cm[0,1])
specificity = (cm[0,0]/(cm[0,0]+cm[1,0]))
print("Confusion Matrix: \n", cm)
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(classification_report(y,y_prediction_values>0))

Confusion Matrix: 
 [[6960    5]
 [  12  188]]
Sensitivity (Recall): 0.9741
Specificity: 0.9983
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      6965
         1.0       0.97      0.94      0.96       200

    accuracy                           1.00      7165
   macro avg       0.99      0.97      0.98      7165
weighted avg       1.00      1.00      1.00      7165



In [ ]:
yv_pred = model.decision_function(Xv)
yv_prediction_values = np.array(yv_pred)
cmv = confusion_matrix(yv, yv_prediction_values>0)
sensitivityv = cmv[1,1]/(cmv[1,1]+cmv[0,1])
specificityv = (cmv[0,0]/(cmv[0,0]+cmv[1,0]))
print("Confusion Matrix: \n", cmv)
print(f"Sensitivity (Recall): {sensitivityv:.4f}")
print(f"Specificity: {specificityv:.4f}")
print(classification_report(yv,yv_prediction_values>0))

Confusion Matrix: 
 [[859  29]
 [  7 153]]
Sensitivity (Recall): 0.8407
Specificity: 0.9919
              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98       888
         1.0       0.84      0.96      0.89       160

    accuracy                           0.97      1048
   macro avg       0.92      0.96      0.94      1048
weighted avg       0.97      0.97      0.97      1048

